In [31]:
:dep polars = {version = "*"}
:dep itertools = {version = "*"}


In [20]:
use polars::prelude::*;

In [21]:
let s: Series = [1, 2, 3].iter().collect();
// Quickly obtain the ChunkedArray wrapped by the Series.
{
    let chunked_array = s.i32().unwrap();
}
s

The type of the variable s was redefined, so was lost.


shape: (3,)
Series: '' [i32]
[
	1
	2
	3
]

In [22]:
// Simple clone of a Series

let s: Series = [1, 2, 3].iter().collect();
// Quickly obtain the ChunkedArray wrapped by the Series.
{
    let inputs = vec![s.clone()];

    fn noop(inputs: &[Series]) -> PolarsResult<Series> {
        let s = &inputs[0];
        Ok(s.clone())
    }

    let result = noop(inputs.as_slice());

    result.unwrap()
}


shape: (3,)
Series: '' [i32]
[
	1
	2
	3
]

In [23]:
// Cast Series to Float64

let s: Series = [1, 2, 3].iter().collect();
// Quickly obtain the ChunkedArray wrapped by the Series.
{
    let inputs = vec![s.clone()];

    fn noop(inputs: &[Series]) -> PolarsResult<Series> {
        let s = &inputs[0].cast(&DataType::Float64)?;
        Ok(s.clone())
    }

    let result = noop(inputs.as_slice());

    result.unwrap()
}

shape: (3,)
Series: '' [f64]
[
	1.0
	2.0
	3.0
]

In [24]:
// Series cast + ChunkedArray + apply

let s: Series = [1, -2, 3].iter().collect();
// Quickly obtain the ChunkedArray wrapped by the Series.
{
    let inputs = vec![s.clone()];

    fn abs_f64(inputs: &[Series]) -> PolarsResult<Series> {
        let s = &inputs[0].cast(&DataType::Float64)?;
        let ca: &Float64Chunked = s.f64()?;
        let out: Float64Chunked = ca.apply(|opt_v: Option<f64>| opt_v.map(|v: f64| v.abs()));
        Ok(out.into_series())
    }

    let result = abs_f64(inputs.as_slice());

    result.unwrap()
}



shape: (3,)
Series: '' [f64]
[
	1.0
	2.0
	3.0
]

In [27]:
// Series cast + ChunkedArray + apply

use polars::prelude::*;


fn mult_f64(inputs: &[Series]) -> PolarsResult<Series> {
    let s = &inputs[0].cast(&DataType::Float64)?;
    let ca: &Float64Chunked = s.f64()?;

//    let out: Float64Chunked = ca.apply(|opt_v: Option<f64>| opt_v.map(|v: f64| v.abs()));

    let out: Float64Chunked = ca
    .into_iter()
    .map(|opt_v| opt_v.map(|v| v * 10.0))
    .collect();

    Ok(out.into_series())
}


let s: Series = [1, -2, 3].iter().collect();
// Quickly obtain the ChunkedArray wrapped by the Series.
{
    let inputs = vec![s.clone()];

    let result = mult_f64(inputs.as_slice());

    result.unwrap()
}

shape: (3,)
Series: '' [f64]
[
	10.0
	-20.0
	30.0
]

In [30]:
/// Iteration over two Series using into_iter and zip

fn mult_two_f64(inputs: &[Series]) -> PolarsResult<Series> {
    // Expecting exactly two inputs
    if inputs.len() != 2 {
        return Err(PolarsError::ComputeError("Expected two input Series".into()));
    }

    // Cast both series to Float64
    let s1 = inputs[0].cast(&DataType::Float64)?;
    let s2 = inputs[1].cast(&DataType::Float64)?;

    // Get ChunkedArray references
    let ca1 = s1.f64()?;
    let ca2 = s2.f64()?;

    // Zip and multiply, preserving nulls
    let out: Float64Chunked = ca1
        .into_iter()
        .zip(ca2.into_iter())
        .map(|(v1, v2)| match (v1, v2) {
            (Some(a), Some(b)) => Some(a * b),
            _ => None,
        })
        .collect();

    Ok(out.into_series())
}

let s: Series = [1, -2, 3].iter().collect();
// Quickly obtain the ChunkedArray wrapped by the Series.
{
    let inputs = vec![s.clone(), s.clone()];

    let result = mult_two_f64(inputs.as_slice());

    result.unwrap()
}

shape: (3,)
Series: '' [f64]
[
	1.0
	4.0
	9.0
]

In [ ]:
// for loop calculation with a vec result. iteration provided by itertools:izip 

use itertools::izip;

fn mult_two_f64_nans(inputs: &[Series]) -> PolarsResult<Series> {
    if inputs.len() != 2 {
        return Err(PolarsError::ComputeError("Expected exactly two input Series".into()));
    }

    let s1 = inputs[0].cast(&DataType::Float64)?;
    let s2 = inputs[1].cast(&DataType::Float64)?;

    if s1.len() != s2.len() {
        return Err(PolarsError::ShapeMismatch(
            format!("Series length mismatch: {} != {}", s1.len(), s2.len()).into(),
        ));
    }

    let ca1 = s1.f64()?;
    let ca2 = s2.f64()?;

    let mut result: Vec<f64> = Vec::with_capacity(ca1.len());

    for (opt_a, opt_b) in izip!(ca1.into_iter(), ca2.into_iter()) {
        // Convert Option<f64> to f64, using NaN as default for None
        let a = opt_a.unwrap_or(f64::NAN);
        let b = opt_b.unwrap_or(f64::NAN);

        result.push(a * b);
    }

    let out = Float64Chunked::from_vec("result".into(), result);
    Ok(out.into_series())
}

let s: Series = [1, -2, 3].iter().collect();
// Quickly obtain the ChunkedArray wrapped by the Series.
{
    let inputs = vec![s.clone(), s.clone()];

    let result = mult_two_f64(inputs.as_slice());

    result.unwrap()
}

shape: (3,)
Series: '' [f64]
[
	1.0
	4.0
	9.0
]

In [54]:
// Macros to iterate over multiple series

use itertools::izip;

#[macro_export]
macro_rules! zip_two_chunkedarrays_into_f64 {
    ($a:expr, $b:expr) => {
        ::itertools::izip!($a.into_iter(), $b.into_iter())
            .map(|(a, b)| (a.unwrap_or(f64::NAN), b.unwrap_or(f64::NAN)))
    };
}

#[macro_export]
macro_rules! izip_two_chunkedarrays_into_f64 {
    ($a:expr, $b:expr) => {
        ($a).into_iter()
            .zip(($b).into_iter())
            .map(|(a, b)| (a.unwrap_or(f64::NAN), b.unwrap_or(f64::NAN)))
    };
}

// this macro must use ident variables to allow for the reuse of the variable names
#[macro_export]
macro_rules! izip_chunkedvars_into_f64 {
    ($($var:ident),+ $(,)?) => {
        ::itertools::izip!($($var.into_iter()),*)
            .map(|($($var),*)| ($($var.unwrap_or(f64::NAN)),*))
    };
}



In [53]:

fn mult_two_f64_nans(inputs: &[Series]) -> PolarsResult<Series> {
    if inputs.len() != 2 {
        return Err(PolarsError::ComputeError("Expected exactly two input Series".into()));
    }

    let s1 = inputs[0].cast(&DataType::Float64)?;
    let s2 = inputs[1].cast(&DataType::Float64)?;

    if s1.len() != s2.len() {
        return Err(PolarsError::ShapeMismatch(
            format!("Series length mismatch: {} != {}", s1.len(), s2.len()).into(),
        ));
    }

    let a = s1.f64()?;
    let b = s2.f64()?;

    let mut result: Vec<f64> = Vec::with_capacity(a.len());

    for (a, b) in izip_chunkedarrays_into_f64!(a, b) {
        result.push(a * b);
    }

    let out = Float64Chunked::from_vec("result".into(), result);
    Ok(out.into_series())
}

let s: Series = [1, -2, 3].iter().collect();
// Quickly obtain the ChunkedArray wrapped by the Series.
{
    let inputs = vec![s.clone(), s.clone()];

    let result = mult_two_f64(inputs.as_slice());

    result.unwrap()
}

shape: (3,)
Series: '' [f64]
[
	1.0
	4.0
	9.0
]